# Autograd - Build
This is the build section of the project, where new functions, classes, etc are created and tested. Finished peices are moved into main where they are then called and used by build.

### Rules:
- No ai used to generate any code, only for reaserch, explinations and code reviews
- Finished peices must be abstract and state clearly if any cases are not covered

## Setup

In [54]:
import numpy as np
import main
from typing import Union, List
from matplotlib import pyplot as plt
import matplotlib_inline
%matplotlib

Using matplotlib backend: module://matplotlib_inline.backend_inline


## Build

In [55]:
class Model():
    """
    
    """
    def __init__(self,
                  input_size : int, hidden_size : int, output_size : int,
                  number_of_layers : int, activation_function, normalisation_function,
                  precision: str = 'float32', random_seed: int = None):
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.number_of_layers = number_of_layers

        self.precision = precision
        self.random_seed = random_seed

        self.activation_function = activation_function
        self.normalisation_function = normalisation_function

        self.activation_function_derivative = main.derivatives.get(
            getattr(self.activation_function, "__name__", None)
        )
        self.normalisation_function_derivative = main.derivatives.get(
            getattr(self.normalisation_function, "__name__", None)
        )


        layers = []
        if self.number_of_layers == 1:
            layers.append(main.LinearLayer(self.input_size, self.output_size,
                                             self.precision, self.random_seed))
        else:
            layers.append(
                main.LinearLayer(self.input_size, self.hidden_size, self.precision, self.random_seed))
            for _ in range(max(0, self.number_of_layers - 2)):
                layers.append(self.activation_function)
                layers.append(main.LinearLayer(self.hidden_size, self.hidden_size, 
                                               self.precision, self.random_seed))
            layers.append(main.LinearLayer(self.hidden_size, self.output_size, 
                                           self.precision, self.random_seed))

        layers.append(self.activation_function)
        layers.append(self.normalisation_function)

        self.modules = layers

        parramaters = {}
        i = 1
        for obj in self.modules:
            try:
                parramaters[f"Layer {i}"] = obj.parramaters
                i += 1
            except:
                continue

        self.parramaters = parramaters

    def forward(self, x: Union[np.ndarray, List, float, int]) -> np.ndarray:
        for module in self.modules:
            if hasattr(module, "forward") and callable(module.forward):
                x = module.forward(x)
            elif callable(module):
                x = module(x)
            else:
                raise TypeError(f"Module - {type(module).__name__} does not have a 'forward' function, or is not callable and so is not supported")
        return x

    

In [56]:
test_inputs = [
    np.array([-1, -0.5, 0]),
    [[0, 1, 2], [2, 0.7, -1]],
    (-2, -0.32, 0),
    "This is a string."
]

model = Model(input_size=3, output_size=2, hidden_size=5,
              number_of_layers=5, activation_function=main.ReLU, normalisation_function=main.Softmax,
              random_seed=1)

In [57]:
print(model.parramaters)

{'Layer 1': {'weights': array([[-0.054,  0.023,  0.51 ,  0.9  , -0.931],
       [-0.712,  0.645,  0.897, -0.502, -0.377],
       [ 0.738, -0.154, -0.454,  0.655, -0.487]], dtype=float32), 'biases': array([-0.182,  0.287,  0.099, -0.829, -0.945], dtype=float32)}, 'Layer 2': {'weights': array([[-0.054,  0.023,  0.51 ,  0.9  , -0.931],
       [-0.712,  0.645,  0.897, -0.502, -0.377],
       [ 0.738, -0.154, -0.454,  0.655, -0.487],
       [-0.182,  0.287,  0.099, -0.829, -0.945],
       [ 0.731,  0.507,  0.675,  0.076,  0.635]], dtype=float32), 'biases': array([-0.341, -0.095,  0.576, -0.753, -0.394], dtype=float32)}, 'Layer 3': {'weights': array([[-0.054,  0.023,  0.51 ,  0.9  , -0.931],
       [-0.712,  0.645,  0.897, -0.502, -0.377],
       [ 0.738, -0.154, -0.454,  0.655, -0.487],
       [-0.182,  0.287,  0.099, -0.829, -0.945],
       [ 0.731,  0.507,  0.675,  0.076,  0.635]], dtype=float32), 'biases': array([-0.341, -0.095,  0.576, -0.753, -0.394], dtype=float32)}, 'Layer 4': {'weig

In [59]:
for t in test_inputs:
    try:
        print(f"Input - {t} - Output - {model.forward(t)}")
        print()
    except:
        print(f"Input - {t} - can not be passed through the model.")

Input - [-1.  -0.5  0. ] - Output - [[0.55897129 0.44102871]]

Input - [[0, 1, 2], [2, 0.7, -1]] - Output - [[0.59192743 0.40807257]
 [0.67192083 0.32807917]]

Input - (-2, -0.32, 0) - Output - [[0.66966686 0.33033314]]

Input - This is a string. - can not be passed through the model.
